# Exotic Options Tutorial

**Cliquet, Autocallable, and Range Accrual**

This tutorial demonstrates pricing three path-dependent exotic products:

1. **Cliquet** – Capped/floored periodic returns (equity/FX)
2. **Autocallable** – Early redemption + coupon + put (equity)
3. **Range Accrual** – Accrual when rate is in range (IR)

All are priced by **Monte Carlo** (no closed form).

**References:** `docs/reference/instruments/exotic_products.md`, `docs/guides/instruments/pricing_exotics.md`

---

## 1. Setup and imports

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
from datetime import date

np.random.seed(42)
print("Imports and path set.")

## 2. Cliquet option

Build the instrument, market data, and run the GBM Monte Carlo pricer.

In [ ]:
from src.instruments.equity.options.cliquet import EquityCliquetOption
from src.pricers.equity.cliquet_gbm_mc import (
    EquityCliquetGbmMcPricer,
    CliquetMarketData,
)

start = date(2025, 1, 1)
end = date(2026, 1, 1)
reset_dates = [date(2025, m, 1) for m in range(1, 13)]

cliquet = EquityCliquetOption(
    underlying_id="SPY",
    notional=1_000_000,
    start_date=start,
    end_date=end,
    reset_dates=reset_dates,
    local_cap=0.03,
    local_floor=-0.01,
    global_cap=0.20,
    global_floor=0.0,
    participation=1.0,
)

market = CliquetMarketData(
    spot=100.0,
    volatility=0.20,
    risk_free_rate=0.05,
    dividend_yield=0.02,
    valuation_date=start,
)

pricer = EquityCliquetGbmMcPricer(n_paths=20_000, seed=42, compute_greeks=True)
result = pricer.price(cliquet, market)

print(f"Cliquet PV: {result.price:,.2f}")
print(f"Std error:  {result.standard_error:,.2f}")
print(f"Delta:      {result.delta:.4f}")
print(f"Vega:       {result.vega:,.2f}")

## 3. Autocallable option

Observation dates, autocall/coupon/put barriers, and coupon rate. Pricer: `EquityAutocallableGbmMcPricer`.

In [ ]:
from src.instruments.equity.options.autocallable import EquityAutocallableOption
from src.pricers.equity.autocallable_gbm_mc import (
    EquityAutocallableGbmMcPricer,
    AutocallableMarketData,
)

maturity = date(2026, 1, 1)
observation_dates = [date(2025, m, 1) for m in range(4, 13)]  # 9 observations

autocall = EquityAutocallableOption(
    underlying_id="SPY",
    notional=1_000_000,
    start_date=start,
    maturity_date=maturity,
    observation_dates=observation_dates,
    autocall_barrier=1.0,   # 100% of initial
    coupon_barrier=0.95,
    put_barrier=0.85,
    coupon_rate=0.08,
)

mkt_autocall = AutocallableMarketData(
    spot=100.0,
    volatility=0.20,
    risk_free_rate=0.05,
    dividend_yield=0.02,
    valuation_date=start,
)

pricer_autocall = EquityAutocallableGbmMcPricer(n_paths=20_000, seed=42, compute_greeks=True)
res_autocall = pricer_autocall.price(autocall, mkt_autocall)

print(f"Autocallable PV: {res_autocall.price:,.2f}")
print(f"Std error:       {res_autocall.standard_error:,.2f}")
print(f"Delta:           {res_autocall.delta:.4f}")

## 4. Range accrual (IR)

Hull-White Monte Carlo pricer; market data includes short-rate parameters.

In [ ]:
from src.instruments.ir.options.range_accrual import IrRangeAccrualNote, ObservationFrequency
from src.pricers.ir.range_accrual_hw_mc import (
    IrRangeAccrualHwMcPricer,
    RangeAccrualMarketData,
)

accrual_start = date(2025, 1, 1)
accrual_end = date(2026, 1, 1)

range_accrual = IrRangeAccrualNote(
    notional=1_000_000,
    start_date=accrual_start,
    maturity_date=accrual_end,
    range_lower=0.02,
    range_upper=0.05,
    accrual_rate=0.08,
    reference_rate_id="USD-SOFR",
    observation_frequency=ObservationFrequency.DAILY,
)

mkt_ra = RangeAccrualMarketData(
    initial_rate=0.04,
    mean_reversion=0.1,
    volatility=0.01,
    long_term_rate=0.04,
    discount_rate=0.04,
    valuation_date=accrual_start,
)

pricer_ra = IrRangeAccrualHwMcPricer(n_paths=15_000, seed=42)
res_ra = pricer_ra.price(range_accrual, mkt_ra)

print(f"Range accrual PV: {res_ra.price:,.2f}")
print(f"Std error:        {res_ra.standard_error:,.2f}")

## 5. Comparison and takeaways

- **Cliquet:** Sensitive to local/global caps and floors and number of resets.
- **Autocallable:** Value driven by autocall probability and coupon; barriers matter.
- **Range accrual:** Depends on rate distribution and range width; Hull-White captures mean reversion.

For more detail and Greeks analysis, see the reference and guide docs.